# 14 · Where do incorrect histogram answers come from?

**Research question:** are wrong acceptances and missed good answers associated with errors already present in the
source-histogram estimate, changes introduced by merging, or cancellation between the two?

This is an offline diagnostic following notebook 13. It does not train models, modify thresholds, correct answers or
benchmark queries. LightGBM is not imported or required. The source sum is a second **approximate estimate**, not ground
truth. The reference truth and all accepted decisions come from existing experiment artifacts.

## Run on Databricks

1. Pull this branch and open this notebook in your personal Git folder. Use the same environment as 13, with pandas,
   NumPy and matplotlib available. There is no LightGBM installation or training cell.
2. Run the configuration cell. Normally **leave `run_dir` blank**: the latest completed **notebook 13** result directory
   under your personal `output_root` is selected by modification time and printed. To reproduce a particular run,
   enter that complete directory containing `run.json`, saved predictions, quality CSV and `outer_splits.csv`.
3. Leave other paths blank initially. The original `rows.csv` is resolved from 13's saved input manifest and its hash
   is checked. Set `rows_file` only if the matching original logger file moved.
4. Run All. Every attempt writes a new `error-origin-...` directory and preserves older results.

When logged source estimates are unavailable, existing `features_v1.source_features` can recompute them from all
verified source histograms and query coordinates. `source_histogram_dir` and `query_dir` are optional relocation
overrides. If those inputs are missing, available analyses still finish with an explicit **partial** status and a missing
input list. Missing sources never become a zero estimate. A changed hash or inconsistent data fails explicitly.

In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

EPSILON = .25
DELTA = .05
EXAMPLES_PER_RULE = 3
KEYS = ['seed','validation_policy','variant','base_rate']
PHYSICAL_KEYS = ['qId','base_rate']
ORIGINS = ['source_good_merged_good','source_good_merged_bad','source_bad_merged_bad',
           'source_bad_merged_good','source_unavailable']


def notebook_defaults():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = context.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace'+root
        output = '/Workspace/Users/'+context.userName().get()+'/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(repo_root=repo,output_root=output,run_dir='',rows_file='',source_histogram_dir='',query_dir='')


DEFAULTS = notebook_defaults()
try:
    for key,value in DEFAULTS.items():
        dbutils.widgets.text(key,value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(),DEFAULTS[key])


def load_settings():
    return {key:str(get_config(key)).strip() for key in DEFAULTS}


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(path,value):
    path = Path(path); temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2,allow_nan=False,default=str))
    temporary.replace(path)


def bool_values(series):
    mapping = {True:True,False:False,'true':True,'false':False,'True':True,'False':False}
    result = series.map(mapping)
    if result.isna().any():
        raise ValueError('Invalid Boolean value in '+str(series.name))
    return result.astype(bool)

## Preserve notebook 13's decisions

The score cutoff is reconstructed from saved **calibration** scores with the original complete-tie rule. It must match
13's saved cutoff and every accepted decision/count. Test truth never selects a new cutoff. Qualification remains
`merged relative error <= 0.25`; the empirical accepted-violation target remains 5%.

The original cohort has exact counts of at least 10, so this diagnostic makes no zero-count quality claim. The outer
cell memberships are checked, and repeated physical estimates must agree across all models and policies before deduplication.

In [ ]:
def calibration_cutoff(scores,errors):
    scores,errors = np.asarray(scores,float),np.asarray(errors,float)
    if not len(scores) or not np.isfinite(scores).all() or not np.isfinite(errors).all():
        raise ValueError('Calibration scores/errors must be nonempty and finite')
    order = np.argsort(scores,kind='stable')
    ordered = scores[order]
    bad = errors[order]>EPSILON
    risk = np.cumsum(bad)/np.arange(1,len(bad)+1)
    endpoints = np.r_[ordered[1:]!=ordered[:-1],True]
    eligible = np.flatnonzero((risk<=DELTA)&endpoints)
    return float(ordered[eligible[-1]]) if len(eligible) else -np.inf


def resolve_run(cfg):
    if cfg['run_dir']:
        source = Path(cfg['run_dir']).expanduser().resolve()
    else:
        candidates = []
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info = json.loads(path.read_text())
            if info.get('status')=='complete' and info.get('study')=='group-validation':
                candidates.append(path)
        if not candidates:
            raise FileNotFoundError('Set run_dir to a completed notebook 13 result directory, containing run.json and saved predictions.')
        source = max(candidates,key=lambda path:path.stat().st_mtime_ns).parent
    prior = json.loads((source/'run.json').read_text())
    if prior.get('status')!='complete' or prior.get('study')!='group-validation':
        raise ValueError('run_dir must be a completed notebook 13 result directory')
    if prior.get('epsilon')!=EPSILON or prior.get('delta')!=DELTA:
        raise ValueError('This diagnostic requires the original 25% error limit and 5% empirical calibration target')
    print('Using notebook 13 run:',source,flush=True)
    return source,prior


def load_and_audit(source,destination):
    names = ['test_predictions.csv','calibration_predictions.csv','quality_by_setting.csv','outer_splits.csv']
    if (source/'preparation/input_manifest.json').is_file():
        names.append('preparation/input_manifest.json')
    identity = {name:sha(source/name) for name in names}
    test = pd.read_csv(source/'test_predictions.csv')
    cal = pd.read_csv(source/'calibration_predictions.csv')
    quality = pd.read_csv(source/'quality_by_setting.csv')
    outer = pd.read_csv(source/'outer_splits.csv')
    for frame in (test,cal):
        if frame.duplicated(KEYS+['qId']).any():
            raise ValueError('Duplicate saved prediction keys')
        frame['accepted'] = bool_values(frame.accepted)
        if not np.isfinite(frame[['truth','estimate','relErr','score_log']].to_numpy(float)).all():
            raise ValueError('Saved predictions contain nonfinite values')
        if (frame.truth<10).any() or (frame.estimate<0).any() or (frame.relErr<0).any():
            raise ValueError('Saved predictions do not match the original exact-count >= 10 cohort')
        if ((frame.qId<0).any() or not np.isfinite(frame.qId).all() or
                not np.equal(frame.qId,np.floor(frame.qId)).all() or
                not np.isfinite(frame.base_rate).all() or (frame.base_rate<0).any()):
            raise ValueError('Invalid saved query IDs or histogram rates')
        if (set(frame.split_kind)!={'cell'} or
                not np.allclose(frame.physical_rate,frame.base_rate,rtol=0,atol=1e-12)):
            raise ValueError('Expected notebook 13 outer cell tests with unchanged physical histogram rates')
        if not np.allclose(frame.relErr,abs(frame.estimate-frame.truth)/frame.truth,rtol=1e-12,atol=1e-12):
            raise ValueError('Saved relative errors do not match estimates and truth')
    if quality.duplicated(KEYS).any() or outer.duplicated(['seed','qId']).any():
        raise ValueError('Duplicate quality or outer split keys')
    for seed,part in outer.groupby('seed'):
        if set(part.split)!={'train','calibration','test'}:
            raise ValueError('Incomplete outer split roles')
        cells=[set(part.loc[part.split==role,'cell']) for role in ('train','calibration','test')]
        if any(cells[i]&cells[j] for i in range(3) for j in range(i)):
            raise ValueError('Outer cell groups overlap')
    quality['accept_none'] = bool_values(quality.accept_none)
    test_settings = test[KEYS].drop_duplicates()
    if set(map(tuple,test_settings.to_numpy()))!=set(map(tuple,quality[KEYS].to_numpy())):
        raise ValueError('Prediction and quality settings differ')
    if set(map(tuple,cal[KEYS].drop_duplicates().to_numpy()))!=set(map(tuple,test_settings.to_numpy())):
        raise ValueError('Calibration and test settings differ')
    audit = []
    quality_index = quality.set_index(KEYS)
    cal_index = {key:frame for key,frame in cal.groupby(KEYS,sort=True)}
    for key,part in test.groupby(KEYS,sort=True):
        row = quality_index.loc[key]
        calibration = cal_index[key]
        split = outer[outer.seed==key[0]]
        if (set(part.qId)!=set(split.loc[split.split=='test','qId']) or
                set(calibration.qId)!=set(split.loc[split.split=='calibration','qId'])):
            raise ValueError('Prediction IDs differ from the frozen outer split')
        if set(part.qId)&set(calibration.qId):
            raise ValueError('Calibration and test query IDs overlap')
        threshold = calibration_cutoff(calibration.score_log,calibration.relErr)
        expected = -np.inf if row.accept_none else float(row.cutoff)
        if not (threshold==expected or np.isclose(threshold,expected,rtol=1e-10,atol=1e-12)):
            raise ValueError('Saved cutoff does not reproduce from calibration alone')
        accepted = part.score_log.to_numpy()<=expected
        cal_accepted = calibration.score_log.to_numpy()<=expected
        n,bad = int(accepted.sum()),int((accepted&(part.relErr.to_numpy()>EPSILON)).sum())
        if (not np.array_equal(accepted,part.accepted.to_numpy()) or n!=row.accepted_count or bad!=row.violation_count
                or len(part)!=row.test_count or len(calibration)!=row.calibration_count
                or not np.array_equal(cal_accepted,calibration.accepted.to_numpy())
                or int(cal_accepted.sum())!=row.calibration_accepted_count
                or int((cal_accepted&(calibration.relErr.to_numpy()>EPSILON)).sum())!=row.calibration_violation_count):
            raise ValueError('Saved gate decisions/counts differ from notebook 13')
        audit.append(dict(**dict(zip(KEYS,key)),test_count=len(part),accepted_count=n,violation_count=bad,
            cutoff=expected if math.isfinite(expected) else np.nan,accept_none=bool(row.accept_none),
            calibration_reproduced=True,decisions_match=True,counts_match=True))
    pd.DataFrame(audit).to_csv(destination/'gate_audit.csv',index=False)
    for name,digest in identity.items():
        if sha(source/name)!=digest:
            raise ValueError('Notebook 13 output changed during reading: '+name)
    # Models/policies share the same physical histogram answers. Verify before deduplicating.
    for column in ('truth','estimate','relErr'):
        extrema = test.groupby(PHYSICAL_KEYS)[column].agg(['min','max'])
        if not np.allclose(extrema['min'],extrema['max'],rtol=1e-12,atol=1e-12):
            raise ValueError('Physical histogram answers differ across models/policies: '+column)
    if (test.groupby(PHYSICAL_KEYS).histogram_key.nunique()!=1).any():
        raise ValueError('A physical query/rate maps to more than one histogram')
    physical = test.sort_values(PHYSICAL_KEYS+KEYS[:-1]).drop_duplicates(PHYSICAL_KEYS)[
        PHYSICAL_KEYS+['histogram_key','truth','estimate','relErr']].reset_index(drop=True)
    manifest = json.loads((source/'preparation/input_manifest.json').read_text()) if 'preparation/input_manifest.json' in identity else {}
    return test,quality,physical,manifest,identity

## Three answers, with separate roles

| Value | Meaning |
|---|---|
| `truth` | Saved exact count used only for offline diagnosis |
| `source_estimate` | Sum of the original source histograms' approximate query answers, before merging |
| `estimate` | The merged histogram answer already evaluated in 13 |

Prefer the verified Java logger's `srcSumEst`. Logger query IDs, histogram rates, true counts and merged estimates must
match saved predictions. Source estimates must be consistent across merge rates. If needed, recomputation requires
**every** expected source histogram, matching query inputs and unchanged source-feature code; no partial source sum is allowed.

In [ ]:
def resolve_input(key,manifest,cfg,source,prior,missing):
    if key not in manifest:
        missing.append(dict(input=key,path='',reason='Not listed in notebook 13 input manifest'))
        return None
    record = manifest[key]
    if key=='run/rows.csv' and cfg['rows_file']:
        candidates = [Path(cfg['rows_file'])]
    elif key.startswith('run/src_') and cfg['source_histogram_dir']:
        candidates = [Path(cfg['source_histogram_dir'])/key.split('/',1)[1]]
    elif key=='data/queries-Ans.txt' and cfg['query_dir']:
        candidates = [Path(cfg['query_dir'])/'queries-Ans.txt']
    else:
        candidates = [Path(record['path']),source/'inputs'/key]
        if prior.get('source_run'):
            candidates.append(Path(prior['source_run'])/'inputs'/key)
    path = next((p.expanduser().resolve() for p in candidates if p.expanduser().is_file()),None)
    if path is None:
        missing.append(dict(input=key,path='; '.join(map(str,candidates)),reason='File unavailable'))
        return None
    if sha(path)!=record['sha256']:
        raise ValueError('Input hash changed since notebook 13: '+str(path))
    return path


def obtain_source_estimates(physical,manifest,cfg,source,prior,destination):
    missing,used,checks = [],{},[]
    if not manifest:
        missing.append(dict(input='preparation/input_manifest.json',path=str(source/'preparation/input_manifest.json'),
                            reason='Source identity metadata unavailable; source estimates cannot be verified'))
    result = physical.copy();result['source_estimate']=np.nan;result['source_estimate_method']='unavailable'
    path = resolve_input('run/rows.csv',manifest,cfg,source,prior,missing)
    if path is not None:
        header = pd.read_csv(path,nrows=0).columns
        required = {'method','mergeRate','qId','gt','est','srcSumEst'}
        if required<=set(header):
            logged = pd.read_csv(path,usecols=list(required))
            if sha(path)!=manifest['run/rows.csv']['sha256']:
                raise ValueError('Logger input changed while reading')
            used['run/rows.csv']=dict(path=str(path),sha256=sha(path))
            for rate in sorted(result.base_rate.unique()):
                old = logged[(logged.method=='ARM')&np.isclose(logged.mergeRate,rate,rtol=0,atol=1e-12)]
                if old.qId.duplicated().any():
                    raise ValueError('Duplicate source logger query IDs at rate '+str(rate))
                index = result.index[result.base_rate==rate]
                part = result.loc[index].merge(old[['qId','gt','est','srcSumEst']],on='qId',how='left',validate='one_to_one',indicator=True)
                have = part['_merge'].eq('both')
                if (have&(~np.isfinite(part['gt'])|~np.isfinite(part.est))).any():
                    raise ValueError('Logger truth/merged estimates contain nonfinite values')
                if ((have&~np.isclose(part['gt'],part.truth,rtol=0,atol=1e-9)).any() or
                        (have&((part.est-part.estimate).abs()>.001)).any()):
                    raise ValueError('Logger truth/merged estimate differs from notebook 13')
                values = part.srcSumEst.to_numpy(float)
                if np.isinf(values).any() or (values[~np.isnan(values)]<0).any():
                    raise ValueError('Logged source estimate is invalid')
                result.loc[index,'source_estimate']=values
                result.loc[index[~np.isnan(values)],'source_estimate_method']='verified logger srcSumEst'
                checks.append(dict(base_rate=rate,physical_cases=len(part),logger_rows_found=int(have.sum()),
                    source_estimates_found=int(np.isfinite(values).sum()),
                    max_merged_estimate_gap=float((part.est-part.estimate).abs().max()) if have.any() else np.nan))
        else:
            missing.append(dict(input='run/rows.csv:srcSumEst',path=str(path),reason='Required logger columns missing: '+','.join(sorted(required-set(header)))))
    # Only missing estimates need recomputation. Every expected source file is mandatory.
    if result.source_estimate.isna().any():
        source_keys = sorted(key for key in manifest if key.startswith('run/src_') and key.endswith('.txt'))
        paths = [resolve_input(key,manifest,cfg,source,prior,missing) for key in source_keys]
        queries = resolve_input('data/queries-Ans.txt',manifest,cfg,source,prior,missing)
        if not source_keys:
            missing.append(dict(input='run/src_*.txt',path='',reason='No source histograms listed; zero-source estimates are forbidden'))
        helper_paths = [Path(cfg['repo_root'])/'ml'/name for name in ('features.py','features_v1.py')]
        helpers_available = all(p.is_file() for p in helper_paths)
        if not helpers_available:
            for p in helper_paths:
                if not p.is_file():missing.append(dict(input='source feature helper',path=str(p),reason='Recomputation helper unavailable'))
        if source_keys and all(p is not None for p in paths) and queries is not None and helpers_available:
            for p in helper_paths:
                expected = prior['source_code_sha256'][p.name]
                if sha(p)!=expected:
                    raise ValueError('Source feature code changed since notebook 13: '+str(p))
            # Staging only verified complete inputs avoids features_v1's empty/partial-source fallback.
            stage = Path(tempfile.mkdtemp(prefix='synopsislake-origin-'))
            (stage/'run').mkdir();(stage/'data').mkdir()
            for key,p in zip(source_keys,paths):
                shutil.copyfile(p,stage/key)
                if sha(stage/key)!=manifest[key]['sha256']:raise ValueError('Source input changed during staging')
                used[key]=dict(path=str(p),sha256=sha(p))
            shutil.copyfile(queries,stage/'data/queries-Ans.txt')
            if sha(stage/'data/queries-Ans.txt')!=manifest['data/queries-Ans.txt']['sha256']:
                raise ValueError('Query input changed during staging')
            used['data/queries-Ans.txt']=dict(path=str(queries),sha256=sha(queries))
            import sys
            sys.path.insert(0,str(Path(cfg['repo_root']).resolve()/'ml'))
            try:
                import features
                import features_v1
                for module in (features,features_v1):
                    if Path(module.__file__).resolve().parent!=Path(cfg['repo_root']).resolve()/'ml':
                        raise ValueError('Another repository is imported; restart Python')
                    importlib.reload(module)
                recomputed = features_v1.source_features(str(stage/'run'),str(stage/'data'))
            except ImportError as error:
                missing.append(dict(input='source feature dependencies',path='',reason=str(error)))
            else:
                if recomputed.qId.duplicated().any() or not np.isfinite(recomputed.src_est).all() or (recomputed.src_est<0).any():
                    raise ValueError('Invalid recomputed source estimates')
                query_truth = pd.read_csv(stage/'data/queries-Ans.txt',header=None).iloc[:,4]
                if (max(result.qId)>=len(query_truth) or
                        not np.allclose(query_truth.iloc[result.qId].to_numpy(),result.truth,rtol=0,atol=1e-9)):
                    raise ValueError('Recomputation queries differ from saved prediction truths')
                lookup = recomputed.set_index('qId').src_est
                values = result.qId.map(lookup)
                existing = result.source_estimate.notna()
                if existing.any() and not np.allclose(result.loc[existing,'source_estimate'],values[existing],rtol=1e-9,atol=.001):
                    raise ValueError('Recomputed source estimates disagree with logger values')
                missing_values = ~existing
                result.loc[missing_values,'source_estimate']=values[missing_values]
                result.loc[missing_values&values.notna(),'source_estimate_method']='recomputed existing source_features'
    known = result[result.source_estimate.notna()]
    if len(known):
        extrema = known.groupby('qId').source_estimate.agg(['min','max'])
        if not np.allclose(extrema['min'],extrema['max'],rtol=1e-12,atol=1e-9):
            raise ValueError('Source estimate changes across merged histogram rates')
    pd.DataFrame(checks,columns=['base_rate','physical_cases','logger_rows_found','source_estimates_found','max_merged_estimate_gap']).to_csv(
        destination/'source_estimate_verification.csv',index=False)
    pd.DataFrame(missing,columns=['input','path','reason']).to_csv(destination/'missing_inputs.csv',index=False)
    save_json(destination/'source_input_manifest.json',used)
    return result,missing,used

## Separate threshold crossings from signed error changes

| Origin class | What is observed |
|---|---|
| `source_good_merged_good` | Both approximate answers meet the 25% limit |
| `source_good_merged_bad` | The source answer meets the limit; the merged answer does not |
| `source_bad_merged_bad` | Both answers break the limit; merging can still improve or worsen the error |
| `source_bad_merged_good` | The source answer breaks the limit; the merged answer meets it |

For positive truth, the signed identity is exact up to floating-point rounding:

`(merged - truth) / truth = (source - truth) / truth + (merged - source) / truth`

**Absolute errors do not add.** Opposing signs can cancel or overshoot. `abs_error_change` reports whether merging
actually improved or worsened absolute error; `interaction` records the signs. A both-bad label is not proof of an
irreducible source error, a theoretical information limit, or that more informative source features cannot help.

These quantities use truth and are **diagnostic only**. They must not become runtime model inputs or new acceptance rules.

Physical summaries count each `(qId, base_rate)` once over the union of saved tests. Model summaries keep seed, validation
policy and variant separate. Repeated queries across seeds/rates are not independent observations. The grouped-validation
policy is primary; the old query-validation policy remains available in the CSVs for comparison.

Representative cases use fixed rules: the worst false accept with a good source, the worst false accept with both answers
bad, the lowest-error false reject when both are good, source-bad/merged-good false rejects, strong cancellation cases,
and both-bad cases where merging improved error. Each rule keeps at most three distinct physical pairs. Ties are broken
by query ID, rate, a fixed compact16/full_source/merged_only priority, then seed. No representative case changes a gate.

In [ ]:
def add_origin_columns(frame):
    result = frame.copy()
    if (not np.isfinite(result[['truth','estimate']].to_numpy(float)).all() or
            (result.truth<=0).any() or (result.estimate<0).any()):
        raise ValueError('Origin diagnostics require positive truth and finite nonnegative estimates')
    source = result.source_estimate.to_numpy(float)
    if np.isinf(source).any() or (source[~np.isnan(source)]<0).any():
        raise ValueError('Source estimates must be nonnegative; missing estimates must remain NaN')
    result['merged_signed_error']=(result.estimate-result.truth)/result.truth
    computed = result.merged_signed_error.abs()
    if 'relErr' in result:
        if not np.allclose(computed,result.relErr,rtol=1e-12,atol=1e-12):
            raise ValueError('Saved merged error differs from the diagnostic estimate')
        result['merged_rel_error']=result.relErr  # Preserve the original classification at a floating-point boundary.
    else:
        result['merged_rel_error']=computed
    result['source_signed_error']=(result.source_estimate-result.truth)/result.truth
    result['source_rel_error']=result.source_signed_error.abs()
    result['merge_signed_change']=(result.estimate-result.source_estimate)/result.truth
    result['abs_error_change']=result.merged_rel_error-result.source_rel_error
    result['source_available']=result.source_estimate.notna()
    result['source_good']=pd.array(np.where(result.source_available,result.source_rel_error<=EPSILON,None),dtype='boolean')
    result['merged_good']=result.merged_rel_error<=EPSILON
    sg = result.source_good.fillna(False).to_numpy(bool); mg = result.merged_good.to_numpy(bool)
    available = result.source_available.to_numpy(bool)
    result['origin_class']=np.select([available&sg&mg,available&sg&~mg,available&~sg&~mg,available&~sg&mg],
        ORIGINS[:4],default='source_unavailable')
    product = result.source_signed_error*result.merge_signed_change
    result['interaction']=np.select([~available,product>0,product<0],
        ['source_unavailable','reinforcing','opposing'],default='zero_component')
    result['cancellation_rel']=np.maximum(0,result.source_signed_error.abs()+result.merge_signed_change.abs()-computed)
    result['merge_abs_effect']=np.select([~available,result.abs_error_change>1e-12,result.abs_error_change < -1e-12],
        ['source_unavailable','worsened','improved'],default='unchanged_within_roundoff')
    result['signed_identity_residual']=result.merged_signed_error-(result.source_signed_error+result.merge_signed_change)
    if not np.allclose(result.loc[available,'merged_signed_error'],
                       result.loc[available,'source_signed_error']+result.loc[available,'merge_signed_change'],rtol=1e-12,atol=1e-12):
        raise ValueError('Signed source-plus-merge error identity failed')
    return result


def origin_summary(physical):
    rows=[]
    for rate,part in physical.groupby('base_rate',sort=True):
        for origin in ORIGINS:
            group = part[part.origin_class==origin]
            known = group[group.source_available]
            rows.append(dict(base_rate=rate,origin_class=origin,physical_cases=len(group),unique_qids=group.qId.nunique(),
                share_of_physical_cases=len(group)/len(part),source_rel_error_median=known.source_rel_error.median(),
                merged_rel_error_median=group.merged_rel_error.median(),merged_rel_error_p90=group.merged_rel_error.quantile(.9),
                abs_error_change_median=known.abs_error_change.median(),reinforcing_cases=int((group.interaction=='reinforcing').sum()),
                opposing_cases=int((group.interaction=='opposing').sum()),merge_worsened_cases=int((group.merge_abs_effect=='worsened').sum()),
                merge_improved_cases=int((group.merge_abs_effect=='improved').sum())))
    return pd.DataFrame(rows)


def decision_metrics(group):
    accepted=group.accepted.to_numpy(bool); good=group.merged_good.to_numpy(bool)
    n=int(accepted.sum());fa=int((accepted&~good).sum());fr=int((~accepted&good).sum())
    return dict(total_cases=len(group),unique_qids=group.qId.nunique(),source_available_cases=int(group.source_available.sum()),
        accepted_count=n,good_accepted_count=n-fa,violation_count=fa,false_accept_count=fa,false_reject_count=fr,
        bad_rejected_count=int((~accepted&~good).sum()),true_good_count=int(good.sum()),
        acceptance=n/len(group) if len(group) else np.nan,violation=fa/n if n else np.nan,
        good_recall=(n-fa)/int(good.sum()) if good.any() else np.nan)


def summarize_decisions(cases):
    summaries,origins=[],[]
    for key,part in cases.groupby(KEYS,sort=True):
        prefix=dict(zip(KEYS,key));totals=decision_metrics(part)
        summaries.append(dict(**prefix,**totals))
        for origin in ORIGINS:
            metrics=decision_metrics(part[part.origin_class==origin])
            origins.append(dict(**prefix,origin_class=origin,**metrics,
                share_of_false_accepts=metrics['false_accept_count']/totals['false_accept_count'] if totals['false_accept_count'] else np.nan,
                share_of_false_rejects=metrics['false_reject_count']/totals['false_reject_count'] if totals['false_reject_count'] else np.nan))
    return pd.DataFrame(summaries),pd.DataFrame(origins)


def representative_cases(cases):
    primary=cases[cases.validation_policy=='cell'].copy()
    primary['variant_order']=primary.variant.map({'compact16':0,'full_source':1,'merged_only':2}).fillna(3)
    rules=[
        ('false_accept_source_good',(primary.model_outcome=='bad_accepted')&(primary.origin_class=='source_good_merged_bad'),'merged_rel_error',False),
        ('false_accept_both_bad',(primary.model_outcome=='bad_accepted')&(primary.origin_class=='source_bad_merged_bad'),'merged_rel_error',False),
        ('false_reject_both_good',(primary.model_outcome=='good_rejected')&(primary.origin_class=='source_good_merged_good'),'merged_rel_error',True),
        ('false_reject_source_bad_merged_good',(primary.model_outcome=='good_rejected')&(primary.origin_class=='source_bad_merged_good'),'source_rel_error',False),
        ('source_bad_merged_good_cancellation',primary.origin_class=='source_bad_merged_good','cancellation_rel',False),
        ('both_bad_but_merge_improved',(primary.origin_class=='source_bad_merged_bad')&(primary.merge_abs_effect=='improved'),'abs_error_change',True),
    ]
    selected=[]
    for rule,mask,sort_column,ascending in rules:
        part=primary[mask].sort_values([sort_column,'qId','base_rate','variant_order','seed','variant'],
            ascending=[ascending,True,True,True,True,True]).drop_duplicates(PHYSICAL_KEYS).head(EXAMPLES_PER_RULE)
        if len(part):selected.append(part.assign(selection_rule=rule))
    columns=['selection_rule',*KEYS,'qId','histogram_key','truth','source_estimate','estimate',
        'source_rel_error','merged_rel_error','source_signed_error','merge_signed_change','merged_signed_error',
        'abs_error_change','interaction','cancellation_rel','score_log','cutoff','accept_none','accepted','model_outcome','origin_class']
    if not selected:return pd.DataFrame(columns=columns)
    return pd.concat(selected,ignore_index=True)[columns]

## Interpret with the right denominators

- `origin_summary.csv` describes unique physical query/histogram pairs.
- `decision_summary.csv` reproduces per-setting accepted, false-accept and false-reject counts.
- `decision_origin_summary.csv` reports each origin group's size, acceptance, accepted violations, good-answer recall,
  and its share of that setting's false accepts/rejects. Zero acceptance has undefined risk.
- `mechanism_summary.csv` distinguishes reinforcing/opposing changes and whether absolute error improved or worsened.
- `representative_cases.csv` includes query/seed/rate, approximate answers, true count, signed errors, score and cutoff.

The figures show physical origin shares, per-setting mistake composition and a deterministic sample of source-versus-merged
error pairs. When source inputs are missing, source-unavailable groups remain explicit and the source-error scatter is skipped.

In [ ]:
def write_analysis(destination,state,physical,summary,origin_decisions):
    known=physical[physical.source_available]
    text=['# Error-origin diagnostic','',
        f"Source estimates available for {len(known):,} of {len(physical):,} unique query/histogram pairs.",
        '','The source sum is an approximate answer, not ground truth. All truth-normalized components are offline diagnostics.',
        'The model still accepts based on the original saved score and calibration cutoff; no model or gate was changed.','']
    if not len(known):
        text += ['## Partial analysis','',
            'No verified source estimates are available. Gate reconstruction and accepted/rejected outcome summaries are complete, '
            'but no claim about source-versus-merge origins can be made. See missing_inputs.csv and configure rows_file '
            'or the complete source_histogram_dir/query_dir inputs.','']
    else:
        text += ['## Unique physical-query origin counts','']
        for rate,part in known.groupby('base_rate'):
            text.append(f'Base rate {rate:.8f}: {len(part):,} pairs with source estimates.')
            for label in ORIGINS[:4]:
                n=int((part.origin_class==label).sum());text.append(f'- {label}: {n:,} ({n/len(part):.2%} of known-source pairs).')
        text += ['','## Per-setting mistakes under the grouped-validation policy','']
        for row in summary[summary.validation_policy=='cell'].itertuples():
            risk=f'{row.violation:.2%}' if math.isfinite(row.violation) else 'undefined'
            text.append(f'- Seed {row.seed}, {row.variant}, rate {row.base_rate:.8f}: '
                f'{row.accepted_count} accepted, {row.false_accept_count} false accepts, {row.false_reject_count} false rejects; risk {risk}.')
        text += ['','## What this can guide','',
            'Inspect each setting in decision_origin_summary.csv. A large source_good_merged_bad share among false accepts '
            'motivates testing information about changes introduced by merging. A large source_bad_merged_bad share motivates '
            'examining source-level uncertainty as well as merge effects. That category can still contain improvements or '
            'reinforcing changes; it does not prove that the source error is irreducible or that a different learner cannot help.',
            'Source_bad_merged_good cases show why source estimates must not be treated as a perfect reference.','']
    text += ['## Limits','',
        'Signed normalized errors satisfy e_merged = e_source + delta_merge. Absolute errors do not add. '
        'An opposing change can reduce error or overshoot; use abs_error_change as well as interaction.',
        'Physical origin summaries count each query/histogram pair once over the union of the saved test sets. '
        'Model summaries keep seed, validation policy and variant separate; repeated cases are not independent trials.',
        'This is exploratory diagnosis on an inspected test set. Truth-based origin flags and normalized merge changes '
        'must not become runtime features or tuned gates. Select future designs using training/calibration data and '
        'validate final claims on fresh held-out queries. The original cohort excludes true counts below 10, including zero.',
        'No model training, raw-data exact execution or runtime/E2E benchmark is performed here.']
    (destination/'analysis_en.md').write_text('\n'.join(text)+'\n')


def make_figures(destination,physical,origin_decisions):
    import matplotlib.pyplot as plt
    colors=dict(zip(ORIGINS,['tab:green','tab:red','tab:orange','tab:blue','lightgray']))
    labels={'source_good_merged_good':'Both acceptable','source_good_merged_bad':'Source acceptable, merged bad',
        'source_bad_merged_bad':'Both bad','source_bad_merged_good':'Source bad, merged acceptable','source_unavailable':'Source unavailable'}
    rates=sorted(physical.base_rate.unique())
    fig,ax=plt.subplots(figsize=(8,4));left=np.zeros(len(rates))
    for origin in ORIGINS:
        values=np.array([100*(physical[physical.base_rate==rate].origin_class==origin).mean() for rate in rates])
        ax.barh(range(len(rates)),values,left=left,color=colors[origin],label=labels[origin]);left+=values
    ax.set(yticks=range(len(rates)),yticklabels=[f'{r:.5f}' for r in rates],xlabel='Unique query/histogram pairs (%)',ylabel='Base rate',
           title='Source and merged answer qualification (25% error limit)')
    ax.legend(fontsize=8,bbox_to_anchor=(0,-.2),loc='upper left',ncol=2)
    fig.tight_layout();fig.savefig(destination/'origin_classes.png',dpi=160,bbox_inches='tight');plt.show();plt.close(fig)
    primary=origin_decisions[origin_decisions.validation_policy=='cell']
    groups=list(primary.groupby(['seed','variant','base_rate'],sort=True))
    fig,axes=plt.subplots(1,2,figsize=(13,max(4,len(groups)*.36)))
    for ax,metric,title in zip(axes,['false_accept_count','false_reject_count'],['False accepts','False rejects']):
        left=np.zeros(len(groups));totals=np.array([g[metric].sum() for _,g in groups])
        for origin in ORIGINS:
            count=np.array([g.loc[g.origin_class==origin,metric].sum() for _,g in groups])
            fraction=np.divide(100*count,totals,out=np.zeros(len(groups),float),where=totals>0)
            ax.barh(range(len(groups)),fraction,left=left,color=colors[origin],label=labels[origin]);left+=fraction
        names=[f'{variant} / seed {seed} / {rate:.4f} (n={int(n)})' for (seed,variant,rate),n in zip([k for k,_ in groups],totals)]
        ax.set(yticks=range(len(groups)),yticklabels=names,xlim=(0,100),xlabel='Share of this setting\'s mistakes (%)',title=title)
        ax.invert_yaxis()
    axes[0].legend(fontsize=7,bbox_to_anchor=(0,-.12),loc='upper left',ncol=2)
    fig.suptitle('Grouped-validation policy; seeds remain separate; n=0 has no composition')
    fig.tight_layout();fig.savefig(destination/'mistake_origins.png',dpi=160,bbox_inches='tight');plt.show();plt.close(fig)
    known=physical[physical.source_available].copy()
    if len(known):
        known['sample_order']=[hashlib.sha256(f'{qid}:{rate:.17g}'.encode()).hexdigest() for qid,rate in known[PHYSICAL_KEYS].itertuples(index=False,name=None)]
        sample=known.sort_values('sample_order').head(5000)
        fig,ax=plt.subplots(figsize=(6,5))
        for origin,part in sample.groupby('origin_class'):
            ax.scatter(np.log1p(part.source_rel_error),np.log1p(part.merged_rel_error),s=7,alpha=.35,color=colors[origin],label=labels[origin])
        boundary=math.log1p(EPSILON);ax.axvline(boundary,color='black',linestyle='--');ax.axhline(boundary,color='black',linestyle='--')
        ax.set(xlabel='log1p(source relative error)',ylabel='log1p(merged relative error)',
            title=f'Error pairs: deterministic {len(sample):,} of {len(known):,} unique pairs')
        ax.legend(fontsize=7);fig.tight_layout();fig.savefig(destination/'source_vs_merged_error.png',dpi=160);plt.show();plt.close(fig)


def export_results(destination):
    with ZipFile(destination/'error_origin_results.zip','w',ZIP_DEFLATED) as archive:
        for path in sorted(destination.rglob('*')):
            if path.is_file() and path.suffix in {'.csv','.json','.md','.png'}:
                archive.write(path,path.relative_to(destination))


def run_error_origin(cfg):
    destination=Path(cfg['output_root']).expanduser().resolve()/('error-origin-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state=dict(status='running',study='error-origin',config=cfg,started_utc=datetime.now(timezone.utc).isoformat(),
        epsilon=EPSILON,delta=DELTA,notes=['No model fitting or gate changes.',
            'Source estimates are approximate, not ground truth or a theoretical information upper bound.',
            'Truth-based normalized changes are offline diagnostics, never runtime features.',
            'Unique physical query/histogram pairs and repeated model/seed cases are reported separately.',
            'All test diagnosis remains exploratory.'])
    save_json(destination/'run.json',state)
    try:
        source,prior=resolve_run(cfg)
        test,quality,physical,manifest,identity=load_and_audit(source,destination)
        state.update(source13_run=str(source),source13_manifest_sha256=sha(source/'run.json'),source13_artifact_sha256=identity,
            seeds=sorted(map(int,test.seed.unique())),validation_policies=sorted(test.validation_policy.unique()),
            variants=sorted(test.variant.unique()),base_rates=sorted(map(float,test.base_rate.unique())))
        physical,missing,used=obtain_source_estimates(physical,manifest,cfg,source,prior,destination)
        physical=add_origin_columns(physical)
        physical.to_csv(destination/'physical_cases.csv',index=False)
        origins=origin_summary(physical);origins.to_csv(destination/'origin_summary.csv',index=False)
        additional=[column for column in physical if column not in test.columns or column in PHYSICAL_KEYS]
        cases=test.merge(physical[additional],on=PHYSICAL_KEYS,how='left',validate='many_to_one')
        cases=cases.merge(quality[KEYS+['cutoff','accept_none']],on=KEYS,how='left',validate='many_to_one')
        cases['model_outcome']=np.select([cases.accepted&cases.merged_good,cases.accepted&~cases.merged_good,
                                         ~cases.accepted&cases.merged_good],
                                        ['good_accepted','bad_accepted','good_rejected'],default='bad_rejected')
        cases.to_csv(destination/'case_predictions.csv',index=False)
        summary,origin_decisions=summarize_decisions(cases)
        summary.to_csv(destination/'decision_summary.csv',index=False)
        origin_decisions.to_csv(destination/'decision_origin_summary.csv',index=False)
        representative_cases(cases).to_csv(destination/'representative_cases.csv',index=False)
        mechanism=physical.groupby(['base_rate','origin_class','interaction','merge_abs_effect'],as_index=False).agg(
            physical_cases=('qId','size'),unique_qids=('qId','nunique'))
        mechanism.to_csv(destination/'mechanism_summary.csv',index=False)
        available=int(physical.source_available.sum())
        state.update(status='complete',analysis_status='complete' if available==len(physical) else 'partial',
            unique_physical_pairs=len(physical),unique_test_qids=int(physical.qId.nunique()),repeated_model_cases=len(cases),
            source_available_pairs=available,source_coverage_fraction=available/len(physical),source_inputs=used,
            missing_input_records=len(missing),finished_utc=datetime.now(timezone.utc).isoformat())
        write_analysis(destination,state,physical,summary,origin_decisions)
        save_json(destination/'run.json',state);export_results(destination)
        print('COMPLETE:',destination,flush=True)
        print('Source analysis:',state['analysis_status'],f'({available:,}/{len(physical):,} unique pairs available)',flush=True)
        return destination,physical,summary
    except BaseException as error:
        state.update(status='failed',error=f'{type(error).__name__}: {error}')
        save_json(destination/'run.json',state)
        print('FAILED; diagnostics preserved:',destination,flush=True)
        raise

## Execute the diagnostic

Success means the saved experiment was faithfully reconstructed. It does not establish a new model improvement.
Check the printed source-analysis status and `missing_inputs.csv` before interpreting source-versus-merge patterns.

In [ ]:
# RUN_ERROR_ORIGIN
config=load_settings()
result_dir,physical_cases,decision_summary=run_error_origin(config)
display(pd.read_csv(Path(result_dir)/'origin_summary.csv')) if 'display' in globals() else print(pd.read_csv(Path(result_dir)/'origin_summary.csv').to_string(index=False))

## Share the result

Send **`error_origin_results.zip`**. It includes the CSVs, three figures when available, English notes, input hashes,
gate audit and run manifest. Source histograms, raw data and model weights are not copied into this archive.

Use the result to formulate one information-design hypothesis for a later sidecar experiment. This is an inspected-test
diagnostic; future feature selection must use training/calibration data, with fresh held-out queries for final claims.

In [ ]:
# PLOT_ERROR_ORIGIN
make_figures(Path(result_dir),physical_cases,pd.read_csv(Path(result_dir)/'decision_origin_summary.csv'))
export_results(Path(result_dir))
print('Upload:',Path(result_dir)/'error_origin_results.zip')